<a href="https://colab.research.google.com/github/SoulaimakH/phishing-detection/blob/main/phishing_detection_pfe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install dependencies
!pip install dnspython python-whois tldextract ipwhois

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.7 MB/s eta 0:00:00


In [4]:
import dns.resolver
import socket
import whois
import tldextract
import datetime
import re
from urllib.parse import urlparse
import math
from collections import Counter
import string

# -----------------------------
# Helper functions
# -----------------------------

# Entropy calculation
def entropy(s):
    p, lns = Counter(s), len(s)
    return -sum(count/lns * math.log2(count/lns) for count in p.values())

# N-grams
def ngrams(s, n=2):
    return [s[i:i+n] for i in range(len(s)-n+1)]

# Numeric percentage
def numeric_percentage(s):
    digits = sum(c.isdigit() for c in s)
    return digits / len(s) * 100 if len(s) > 0 else 0

# Longest word (letters only)
def longest_word(s):
    words = re.findall(r'[a-zA-Z]+', s)
    return max(words, key=len) if words else ""

# Obfuscation checks
def check_hex(s): return int(bool(re.search(r'%[0-9a-fA-F]{2}', s)))
def check_at(s): return int('@' in s)
def check_decimal(s): return int(bool(re.search(r'\b\d{2,}\b', s)))
def check_octal(s): return int(bool(re.search(r'\\[0-7]{1,3}', s)))

# Typosquatting placeholder (optional, can use Levenshtein distance)
def typos_similar(domain, popular_domains=['google.com','apple.com']):
    # return list of tuples (domain, similarity score)
    return [(d, 0) for d in popular_domains]

# -----------------------------
# Main feature extraction function
# -----------------------------
def extract_features(domain_name):
    features = {}

    # Normalize domain
    domain_name = domain_name.lower().strip()

    # ---------------------
    # DNS / Network info
    # ---------------------
    try:
        answers = dns.resolver.resolve(domain_name, 'A')
        ip = answers[0].to_text()
        ttl = answers.rrset.ttl
    except:
        ip = None
        ttl = 0

    features['IP'] = ip
    features['TTL'] = ttl

    # Country & ASN lookup (optional, requires geoip2 or ipwhois)
    try:
        from ipwhois import IPWhois
        obj = IPWhois(ip)
        res = obj.lookup_rdap()
        features['ASN'] = res.get('asn', None)
        features['Country'] = res.get('asn_country_code', None)
    except:
        features['ASN'] = None
        features['Country'] = None

    # Name server count
    try:
        ns = dns.resolver.resolve(domain_name, 'NS')
        features['Name_Server_Count'] = len(ns)
    except:
        features['Name_Server_Count'] = 0

    # ---------------------
    # WHOIS info
    # ---------------------
    try:
        w = whois.whois(domain_name)
        features['Registrant_Name'] = w.get('name', None)
        features['Organization'] = w.get('org', None)
        creation_date = w.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]
        features['Creation_Date_Time'] = creation_date
        features['Domain_Age'] = (datetime.datetime.now() - creation_date).days if creation_date else 0
    except:
        features['Registrant_Name'] = None
        features['Organization'] = None
        features['Creation_Date_Time'] = None
        features['Domain_Age'] = 0

    # ---------------------
    # Domain structure
    # ---------------------
    ext = tldextract.extract(domain_name)
    features['sld'] = ext.domain
    features['tld'] = ext.suffix
    features['subdomain'] = ext.subdomain
    features['Domain_Name'] = domain_name
    features['len'] = len(domain_name)
    features['longest_word'] = longest_word(domain_name)

    # ---------------------
    # Text / NLP features
    # ---------------------
    features['entropy'] = entropy(domain_name)
    features['1gram'] = list(domain_name)
    features['2gram'] = ngrams(domain_name, 2)
    features['3gram'] = ngrams(domain_name, 3)
    features['char_distribution'] = dict(Counter(domain_name))
    features['numeric_percentage'] = numeric_percentage(domain_name)
    features['typos'] = typos_similar(domain_name)

    # ---------------------
    # Obfuscation features
    # ---------------------
    features['hex_8'] = check_hex(domain_name)
    features['hex_32'] = check_hex(domain_name)  # placeholder, can refine
    features['dec_8'] = check_decimal(domain_name)
    features['dec_32'] = check_decimal(domain_name)
    features['oc_8'] = check_octal(domain_name)
    features['oc_32'] = check_octal(domain_name)
    features['obfuscate_at_sign'] = check_at(domain_name)
    features['puny_coded'] = int(domain_name.startswith('xn--'))

    return features

# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    domain = "www.ksylitol.com"
    features = extract_features(domain)
    for k,v in features.items():
        print(f"{k}: {v}")

IP: 91.228.196.192
TTL: 14400
ASN: 41079 198414
Country: PL
Name_Server_Count: 0
Registrant_Name: None
Organization: None
Creation_Date_Time: None
Domain_Age: 0
sld: ksylitol
tld: com
subdomain: www
Domain_Name: www.ksylitol.com
len: 16
longest_word: ksylitol
entropy: 3.327819531114783
1gram: ['w', 'w', 'w', '.', 'k', 's', 'y', 'l', 'i', 't', 'o', 'l', '.', 'c', 'o', 'm']
2gram: ['ww', 'ww', 'w.', '.k', 'ks', 'sy', 'yl', 'li', 'it', 'to', 'ol', 'l.', '.c', 'co', 'om']
3gram: ['www', 'ww.', 'w.k', '.ks', 'ksy', 'syl', 'yli', 'lit', 'ito', 'tol', 'ol.', 'l.c', '.co', 'com']
char_distribution: {'w': 3, '.': 2, 'k': 1, 's': 1, 'y': 1, 'l': 2, 'i': 1, 't': 1, 'o': 2, 'c': 1, 'm': 1}
numeric_percentage: 0.0
typos: [('google.com', 0), ('apple.com', 0)]
hex_8: 0
hex_32: 0
dec_8: 0
dec_32: 0
oc_8: 0
oc_32: 0
obfuscate_at_sign: 0
puny_coded: 0
